In [1]:
import cv2 as cv
import numpy as np
import os
from video_to_frames import video_to_frames

**Parameters**

In [2]:
# Frame params
FRAME_HEIGHT = 480
FRAME_WIDTH = 640

num_bins_x = 4
num_bins_y = 3

# Keypoint params
response_cutoff = 0.000

# Number of frames to process
n_frames = 10

In [3]:
# Create output folder
output_folder = "orb_keypoints"
os.makedirs(output_folder, exist_ok=True)

# Initiate ORB detector
orb = cv.ORB_create(nfeatures=500)

# Storing keypoint data
kp_list = []

# Process images
for i in range(1, n_frames + 1):
    img_path = rf"Images\img{i:04d}.png"
    
    img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
    
    if img is None:
        print(f"Failed to load {img_path}")
        continue
    
    # Detect and compute keypoints
    kp = orb.detect(img, None)
    kp, des = orb.compute(img, kp)

    # Filter keypoints by response
    kp_filtered = [k for k in kp if k.response >= response_cutoff]

    # Save raw keypoints
    kp_list.append(kp_filtered)
    
    # Draw keypoints
    img_kp = cv.drawKeypoints(img, kp_filtered, None, color=(0,255,0), flags=0)
    
    # Draw grid
    bin_width = FRAME_WIDTH // num_bins_x
    bin_height = FRAME_HEIGHT // num_bins_y
    
    # Vertical lines
    for x in range(0, FRAME_WIDTH + 1, bin_width):
        cv.line(img_kp, (x, 0), (x, FRAME_HEIGHT), (0, 0, 255), 2)
    
    # Horizontal lines
    for y in range(0, FRAME_HEIGHT + 1, bin_height):
        cv.line(img_kp, (0, y), (FRAME_WIDTH, y), (0, 0, 255), 2)
    
    # Save result
    output_path = os.path.join(output_folder, f'kp_img{i:04d}.png')
    cv.imwrite(output_path, img_kp)

print(f"\nAll images saved to: {output_folder}")


All images saved to: orb_keypoints


**Bin point count**

In [70]:
x_data = []
y_data = []
for frame in kp_list:
    x_data.extend([kp.pt[0] for kp in frame if kp.response >= response_cutoff])
    y_data.extend([kp.pt[1] for kp in frame if kp.response >= response_cutoff])

hist, x_edges, y_edges = np.histogram2d(x=x_data, y=y_data, bins=[num_bins_x, num_bins_y], range=[[0, FRAME_HEIGHT], [0, FRAME_WIDTH]])

for i in range(len(x_edges) - 1):
    for j in range(len(y_edges) - 1):
        print(f"Point count [[{x_edges[i]:.2f}, {x_edges[i+1]:.2f}], \t [{y_edges[j]:.2f}, {y_edges[j+1]:.2f}]]: \t{hist[i, j]}")

print(hist)


Point count [[0.00, 120.00], 	 [0.00, 213.33]]: 	561.0
Point count [[0.00, 120.00], 	 [213.33, 426.67]]: 	132.0
Point count [[0.00, 120.00], 	 [426.67, 640.00]]: 	0.0
Point count [[120.00, 240.00], 	 [0.00, 213.33]]: 	121.0
Point count [[120.00, 240.00], 	 [213.33, 426.67]]: 	559.0
Point count [[120.00, 240.00], 	 [426.67, 640.00]]: 	6.0
Point count [[240.00, 360.00], 	 [0.00, 213.33]]: 	76.0
Point count [[240.00, 360.00], 	 [213.33, 426.67]]: 	84.0
Point count [[240.00, 360.00], 	 [426.67, 640.00]]: 	21.0
Point count [[360.00, 480.00], 	 [0.00, 213.33]]: 	736.0
Point count [[360.00, 480.00], 	 [213.33, 426.67]]: 	135.0
Point count [[360.00, 480.00], 	 [426.67, 640.00]]: 	2.0
[[561. 132.   0.]
 [121. 559.   6.]
 [ 76.  84.  21.]
 [736. 135.   2.]]


In [ ]:
from frame_processing_and_encoding import FrameProcessor

####### Testing ########

# Initialize processor
thresholds = (0, 5, 25, 50, 100, 500)
processor = FrameProcessor(threshold_edges=thresholds)

# Loop through n_frame images
for idx in range(1, n_frames+1):
    img_path = rf"Images\img{idx:04d}.png"
    
    # Returns spike train per frame
    spike_train = processor.process_and_encode_frame(img_path)
    
    print(f"Frame {idx:04d} results:")
    print(f"Spike Train: {spike_train}")
    print("-" * 30)

Frame 0001 results:
Spike Train: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
------------------------------
Frame 0002 results:
Spike Train: [0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0]
------------------------------
Frame 0003 results:
Spike Train: [0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0]
------------------------------
Frame 0004 results:
Spike Train: [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
------------------------------
Frame 0005 results:
Spike Train: [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]
------------------------------
Frame 0006 results:
Spike Train: [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
------------------------------
Frame 0007 results:
Spike Train: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
------------------------------
Frame 0008 results:
Spike Train: [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
------------------------------
Frame 0009 results:
Spike Train: [1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0]
------------------------------
Frame 0010 results:
Spike Train: [0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0]
---------------------